# 00. Por que randomizar?

Antes de usar a biblioteca, vale entender o problema que a randomização resolve. Este notebook mostra, com uma simulação pequena, por que comparar tratados e controles **sem** randomizar leva a conclusões erradas.

## O problema fundamental

Cada unidade tem dois resultados potenciais, `Y(0)` e `Y(1)`, mas só observamos um. Não dá para medir o efeito numa unidade isolada. A saída é estimar o efeito **médio** comparando grupos, e aí entra o risco: se quem recebe o tratamento é sistematicamente diferente de quem não recebe, a diferença de médias mistura o efeito com essa diferença.

## Quando a escolha do tratamento é enviesada

Simulamos um confundidor `x` que afeta tanto o resultado quanto a chance de a unidade escolher o tratamento (auto-seleção). O efeito verdadeiro é `1.0`.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)
n = 2000

x = rng.normal(size=n)                 # confundidor: afeta tratamento e resultado
tau = 1.0                              # efeito verdadeiro
y0 = 2.0 * x + rng.normal(size=n)      # Y(0) depende de x
y1 = y0 + tau                          # Y(1)

# Auto-selecao: quem tem x alto tende a escolher o tratamento
p_self = 1.0 / (1.0 + np.exp(-x))
t_self = (rng.uniform(size=n) < p_self).astype(int)
y_self = np.where(t_self == 1, y1, y0)

naive = y_self[t_self == 1].mean() - y_self[t_self == 0].mean()
print(f"Estimativa ingenua (com confundimento): {naive:.3f}  | verdadeiro: {tau}")

A estimativa ingênua fica **bem acima** de `1.0`: os tratados têm `x` mais alto e, portanto, `Y(0)` mais alto de partida. A diferença de médias atribuiu ao tratamento algo que era do confundidor.

## Randomização quebra o confundimento

Agora atribuímos o tratamento **ao acaso** com `CRD`, ignorando `x`. Como a atribuição é independente de `x`, os dois grupos ficam comparáveis e a diferença de médias estima o efeito verdadeiro.

In [ ]:
from skxperiments.core.assignment import CRDAssignment
from skxperiments.design.crd import CRD
from skxperiments.estimators.difference_in_means import DifferenceInMeans

df = pd.DataFrame({"x": x})
design = CRD(p=0.5, seed=7)
assignment = design.randomize(df)

t = assignment.data_[assignment.treatment_col_].to_numpy()
data = assignment.data_.copy()
data["y"] = np.where(t == 1, y1, y0)
assignment = CRDAssignment(
    data=data, treatment_col=assignment.treatment_col_, design=design, seed=7
)

ate = DifferenceInMeans(outcome_col="y").fit(assignment).estimate().ate
print(f"Estimativa randomizada: {ate:.3f}  | verdadeiro: {tau}")

## O que aprendemos

- Sem randomização, a diferença de médias confunde o efeito com diferenças pré-existentes entre os grupos.
- Randomizar torna os grupos comparáveis **por construção**, sem precisar medir ou modelar o confundidor.
- É por isso que a biblioteca começa pelo **desenho**: o `CRD` (e os outros designs) é quem garante essa comparabilidade.

**Próximo:** `01. Seu primeiro experimento` mostra o fluxo completo de ponta a ponta.